## First Part

The first part is explaining how reliable a test error estimate is and how many test samples we need to estimate the true error of a classifier.

### 1. Two different Errors: Empirical vs Population

Suppose you already trained a classifier $f$.

You now test it on a new dataset
$$
\mathcal{D}=\left\{\left(\mathbf{x}^{(i)}, y^{(i)}\right)\right\}_{i=1}^n
$$
These samples were not used for training.

#### 1 Empirical Test Error

The empirical error is simply the fraction of mistakes on the test set:
$$
\epsilon_{\mathcal{D}}(f)=\frac{1}{n} \sum_{i=1}^n 1\left(f\left(x^{(i)}\right) \neq y^{(i)}\right)
$$
Meaning
For each example:
$$
\mathbf{1}(f(x) \neq y)
$$
is an indicator variable
- =1→ classifier made a mistake
- $=0 \rightarrow$ classifier correct

#### 2 Population Error (True Error)

The real quantity we care about is
$$
\epsilon(f)=E_{(x, y) \sim P}[\mathbf{1}(f(x) \neq y)]
$$
Meaning:

If we sample infinitely many examples from the real world distribution $P(X, Y)$, what fraction would the classifier misclassify?

Equivalent integral form:
$$
\epsilon(f)=\iint 1(f(x) \neq y) p(x, y) d x d y
$$

We cannot observe $\epsilon(f)$ directly.

Why?

Because we don't know the full data distribution $P(X, Y)$.

We only have samples.

So:

Empirical error $\epsilon_D(f)$

is an estimator of

True error $\epsilon(f)$

We can view $\epsilon_{\mathcal{D}}(f)$ as a statistical estimator of the population error $\epsilon(f)$. Moreover, because our quantity of interest $\epsilon(f)$ is an expectation (of the random variable $\mathbf{1}(f(X) \neq Y)$ ) and the corresponding estimator $\epsilon_{\mathcal{D}}(f)$ is the sample average, estimating the population error is simply the classic problem of mean estimation.

### 2. Why Test Error Approximates True Error

Define
$$
Z_i=1\left(f\left(X_i\right) \neq Y_i\right)
$$
Each $Z_i$ :
- takes value 0 or 1
- indicates mistake or not
Then
$$
\epsilon_D(f)=\frac{1}{n} \sum Z_i
$$
This is the sample mean.

And
$$
E\left[Z_i\right]=\epsilon(f)
$$
So we are estimating the mean of a random variable.

### 3. Central Limit Theorem Result

Central Limit Theorem says:

If
$$
\hat{\mu}=\frac{1}{n} \sum a_i
$$
then
$$
\hat{\mu} \approx N\left(\mu, \sigma^2 / n\right)
$$
for large $n$.

Apply it here
$$
\epsilon_D(f) \approx N\left(\epsilon(f), \sigma^2 / n\right)
$$
Standard deviation:
$$
\sigma / \sqrt{n}
$$
Key takeaway

Error decreases at rate
$$
O(1 / \sqrt{n})
$$
Interpretation

If you want twice the precision

you need
$$
4 x
$$
more samples.

### 4. Variance of the Error Indicator

Recall
$$
Z=1(f(X) \neq Y)
$$
This is a Bernoulli variable.

So
$$
P(Z=1)=\epsilon(f)
$$
Variance of Bernoulli:
$$
\sigma^2=p(1-p)
$$
Thus
$$
\sigma^2=\epsilon(f)(1-\epsilon(f))
$$
Maximum variance

The function
$$
p(1-p)
$$
is largest when
$$
p=0.5
$$
Then
$$
\sigma^2=0.25
$$
Therefore
$$
\sigma \leq 0.5
$$

### 5. Standard Deviation of Test Error

CLT says:
$$
s t d\left(\epsilon_D\right)=\frac{\sigma}{\sqrt{n}}
$$
Worst case:
$$
\sigma=0.5
$$
So
$$
s t d\left(\epsilon_D\right) \leq \sqrt{0.25 / n}
$$

### 6. Example: How Many Samples Do We Need?

Suppose we want:
$$
\operatorname{std}\left(\epsilon_D\right)=0.01
$$
Then
$$
\sqrt{0.25 / n}=0.01
$$
Square both sides:
$$
0.25 / n=0.0001
$$
Solve:
$$
n=2500
$$
But 95\% confidence interval is roughly
$$
2 \times s t d
$$
So we want std to be around 0.005

Thus
$$
n \approx 10000
$$

Interpretation

With 10,000 test samples

you can estimate error within
$$
\pm 0.01
$$
with $\sim 95 \%$ confidence.

This is why datasets like
- CIFAR
- ImageNet
- MNIST
often have 10k test examples.

### 7. Why Tiny Improvements Can Be Misleading

If test set size is 10,000:
- error improvement = 0.005
could easily be noise.

Because the statistical uncertainty is about:
$$
\pm 0.01
$$
So many ML papers claiming 0.005 improvement might not be statistically meaningful.

### 8. Problem with the CLT Analysis

CLT is asymptotic.

Meaning:

It only holds when
$$
n \rightarrow \infty
$$
But real datasets are finite.

So we want finite sample guarantees.

### 9. Hoeffding Inequality

For bounded variables (like 0/1 variables):
$$
P\left(\epsilon_D(f)-\epsilon(f) \geq t\right)<\exp \left(-2 n t^2\right)
$$
Meaning

Probability that test error overestimates true error by $t$ is exponentially small.

### 10. Solve Example

Suppose we want
$$
P\left(\epsilon_D-\epsilon>0.01\right)<0.05
$$
Hoeffding gives approximately
$$
n \approx 15000
$$

## Second Part

The second part is less about math and more about research methodology—specifically how test sets can be accidentally misused and why that leads to false scientific conclusions.

### 1. What We Just Learned

From the previous section:

For a fixed classifier $f$ :
- empirical error
$$
\epsilon_{\mathcal{D}}(f)
$$
- true population error
$$
\epsilon(f)
$$
We know:
$$
\epsilon_{\mathcal{D}}(f) \approx \epsilon(f)
$$
and we even know how accurate the estimate is using
- CLT
- Hoeffding bounds.

This analysis assumes the classifier $f$ is fixed.

That assumption is the key to everything that follows.

### 2. A Perfect Experimental Setup

Suppose you follow all the rules of good ML practice.

You:
1. Split dataset into
- training set
- validation set
- test set
2. Use training set to train model
3. Use validation set for
- hyperparameter tuning
- architecture selection
4. Touch the test set only once.
You evaluate:
$$
f_1
$$
and report
$$
\epsilon_{\mathcal{D}}\left(f_1\right)
$$
with confidence interval.

Everything is statistically valid.

So far, perfect.

### 3. Then Something Happens (Real Research)

The lecture describes a very realistic situation.

At 3am you get a new idea:

A new model
$$
f_2
$$
You:
- code it
- tune hyperparameters on validation set

Now the validation results look better than $f_1$.

So naturally you want to test it.

But suddenly you realize:
* You already used the test set.

Now there is no unbiased test set left.

Even though the original test set $\mathcal{D}$ is still sitting on your server, you now face two formidable problems. First, when you collected your test set, you determined the required level of precision under the assumption that you were evaluating a single classifier $f$. However, if you get into the business of evaluating multiple classifiers $f_1, \ldots, f_k$ on the same test set, you must consider the problem of false discovery. Before, you might have been $95 \%$ sure that $\epsilon_{\mathcal{D}}(f) \in \epsilon(f) \pm 0.01$ for a single classifier $f$ and thus the probability of a misleading result was a mere $5 \%$. With $k$ classifiers in the mix, it can be hard to guarantee that there is not even one among them whose test set performance is misleading.

### 4. First Problem: Multiple Hypothesis Testing

Probability of misleading result increases

Example:

If each test has 5\% error probability.

Testing 20 models means:

Probability at least one is misleading:
$$
1-(0.95)^{20}
$$
Compute roughly:
$$
(0.95)^{20} \approx 0.36
$$
So:
$$
1-0.36=0.64
$$
So there is $64 \%$ chance that at least one result is misleading.

Meaning:

You might pick the best-looking model simply due to noise.

This is the classic problem of multiple hypothesis testing.

### 5. Second Problem: Adaptive Overfitting

There is an even worse issue.

The new model $f_2$ was designed after seeing test results for $f_1$.

So the modeler has now used information from the test set.

That means the test set is no longer independent.

Why independence matters

Our earlier analysis assumed:
$$
f \text { independent of test data }
$$
But now:
$$
f_2=f_2\left(\text { test results of } f_1\right)
$$
So the classifier depends on the test set indirectly.

Example intuition

Imagine repeatedly testing models and slightly adjusting them until they look good on the test set.

Eventually you get a model that fits the noise of the test set.

But it won't generalize.

This is called adaptive overfitting

### 6. Why This Became Important Recently

This problem became serious in ML because:

Researchers often evaluate hundreds of models on the same benchmark.

Examples:
- ImageNet
- GLUE
- CIFAR
- Kaggle competitions

Each new paper:
- tries slightly different architectures
- evaluates on the same test set.

Eventually models start to overfit the benchmark.

Even if nobody intentionally cheats.

### 7. What Should Researchers Do?

#### 1. Create real test sets

Keep them separate from development process. Only evaluate on test set when ready to publish.

#### 2. Account for multiple testing

When many models are tested: 

confidence intervals must be adjusted.

Methods include:
- Bonferroni correction
- FDR control.

### 8. Best Practice for Benchmarks

When running benchmark challenges:

Use multiple test sets.

Example workflow:
```
Round 1:
    train
    validation
    test_1
Round 2:
    train
    validation
    test_2
```
Old test sets become validation sets.

New hidden test sets are introduced.

### 9. The Core Statistical Lesson

The entire issue comes from this fact:

Earlier guarantees assumed:
$$
f \text { fixed before test data }
$$
But in real research:
$$
f \text{ evolves based on test feedback }
$$
Thus:
$$
\epsilon_D(f)
$$
becomes optimistically biased.

## Third Part

This part is the bridge from practical ML to statistical learning theory. It is asking:

Can we predict generalization before seeing a test set?

Previously everything depended on test sets. Now learning theory tries to answer:

Why should training error approximate test error?

### 1. Why Test Sets Feel Unsatisfying

Problem 1: True test sets rarely exist

If you use a benchmark dataset (e.g. MNIST, ImageNet):
- thousands of researchers have already evaluated models on it
- information about the test set has leaked into the research community

So the test set is no longer perfectly "fresh".

Problem 2: You can only use it once
After you evaluate:

$$
f_1
$$

on the test set, the test set is contaminated.
Now when you build

$$
f_2
$$

you already know something about the test distribution.
So your test set is no longer unbiased.

Problem 3: Test sets only give post hoc validation
They answer:
Did the model generalize?
But they do not answer:
Should we expect this model to generalize?
This motivates statistical learning theory.

### 2. Goal of Statistical Learning Theory

Statistical learning theory tries to explain:
Why models trained on finite data generalize.
More precisely, it tries to bound the generalization gap:

$$
\text { generalization gap }=\epsilon(f)-\epsilon_S(f)
$$

where
- $\epsilon_S(f)=$ training error
- $\epsilon(f)=$ population error.

### 3. Key Difference From Previous Secion

Previously:
- classifier $f$ was fixed
- dataset used only for evaluation

So estimating error was easy.
Now:

$$
f_S
$$

is trained using the same dataset $S$.
This creates selection bias.

Why this matters
Training algorithm chooses

$$
f_S=\arg \min _{f \in \mathcal{F}} \epsilon_S(f)
$$


So we deliberately pick the classifier with lowest training error.
This means:

$$
\epsilon_S\left(f_S\right)
$$

is optimistically biased.

### 4. The Real Question

We want to know:

$$
\epsilon\left(f_S\right) \approx \epsilon_S\left(f_S\right) ?
$$


Meaning:
Does training error $\approx$ test error?

This is the core question of learning theory.

### 5. Why This is Hard

Suppose models come from a class:

$$
\mathcal{F}
$$


Example:
- linear classifiers
- neural networks
- decision trees.

Even if each individual model generalizes well, problems arise when we choose among many models.

Example intuition
Imagine testing 100 classifiers.
Even if each one's error estimate is accurate with $95 \%$ probability, there is a good chance one classifier's estimate is very wrong.

If we pick that one (because it has smallest training error), we get massive overfitting.

### 6. Infinite Model Classes

Things get worse.
For most ML models:

$$
|\mathcal{F}|=\infty
$$


Examples:
- linear models (continuous parameters)
- neural networks
- SVMs.

So the number of possible classifiers is infinite.

### 7. The Idea of Uniform Convergence

Learning theory proposes a stronger guarantee.
Instead of:

$$
\epsilon_S(f) \approx \epsilon(f)
$$

for one classifier, we want

$$
\epsilon_S(f) \approx \epsilon(f)
$$

for all classifiers simultaneously.
This is called
uniform convergence
Formally:
With probability $1-\delta_{\text {, }}$

$$
\left|\epsilon_S(f)-\epsilon(f)\right|<\alpha
$$

for every $f \in \mathcal{F}$.

### 8. But Uniform Convergence Cannot Hold for All Model Classes

Example: memorization machines

A model that memorizes training data:
- training error $=0$
- population accuracy = random guessing.

Thus:

$$
\epsilon_S(f)=0
$$

but

$$
\epsilon(f)=0.5
$$


Huge gap.
So uniform convergence fails.

Why?
Because the model class is too flexible.

### 9. Bias-Variance Tradeoff Appears

Flexible models
- fit training data well
- high variance
- risk overfitting

Examples:
- deep networks
- high-degree polynomials.

Rigid models
- generalize better
- but may underfit

Examples:
- linear models.

Learning theory tries to quantify this tradeoff.

### 10. The VC Dimension

This is where Vapnik-Chervonenkis theory enters.

They introduced a measure of model complexity:

VC dimension

Definition (informal)

VC dimension =

largest number of points that the model class can shatter.

What does "shatter" mean?
A set of points is shattered if:
For every possible binary labeling, some model in the class can realize it.

Example: Linear classifiers in 2D
Three points:
You can label them any way and draw a line to separate them.
So:

$$
V C=3
$$

Four points:
Some labelings cannot be separated by a line.
Thus not shattered.

General result:

Linear models in $d$ dimensions:

$$
V C=d+1
$$

### 11. The VC Generalization Bound

One of the main results of learning theory:

$$
P\left(R[p, f]-R_{e m p}[X, Y, f]<\alpha\right) \geq 1-\delta
$$

when

$$
\alpha \geq c \sqrt{\frac{V C-\log \delta}{n}}
$$

Meaning
With probability at least $1-\delta$ :
generalization gap
is bounded by
$\alpha$

Important variables
n
dataset size

$$
V C
$$

model complexity

$$
\delta
$$

failure probability

$\alpha$ maximum allowed generalization gap.

### 12. Interpretation

Ignoring constants:

$$
\text { generalization gap } \sim \sqrt{\frac{V C}{n}}
$$

Three insights
More data helps

$$
\propto 1 / \sqrt{n}
$$


Same rate we saw earlier.

Complex models generalize worse

$$
\propto \sqrt{V C}
$$

Tradeoff

To control generalization:
- increase $n$
- or decrease model complexity.

### 13. Why This Theory is Often Pessimistic

In practice:
Deep neural networks have

$$
V C=\text { huge }
$$


Yet they still generalize well.
VC theory would predict we need astronomically large datasets.
But empirically we don't.
Thus VC bounds are often very loose.